# 🌿 TerraSpectra: Multi-Temporal Disease Progression, Production FastAPI Microservice & MLOps Governance
## Day 10 — Spatiotemporal Phenology Trajectories, RESTful Inference Engine, Sensor Drift Auditing & Grand Capstone Defense

**Module:** Multi-Temporal Crop Pathology, Production Edge-Cloud RESTful Serving & MLOps Governance  
**Author:** Rohit (TerraSpectra ML & Remote Sensing Team)  
**Branch:** `Rohit-EDA-Analysis`  
**Milestone:** Day 10 — Multi-Temporal Forecasting, FastAPI Microservice & MLOps Governance  
**Date:** November 26, 2026  

---

### 1. Research Scope & Day 10 Objectives
Across the 10-day research program, TerraSpectra has systematically transformed raw hyperspectral remote sensing into an enterprise-grade precision agriculture platform. **Day 10** represents the operational capstone:

1. **Multi-Temporal Crop Phenology & Disease Progression Dynamics:**
   - Crop foliar pathogens do not exist as static snapshots. Pathogen proliferation is dynamically coupled to thermal time (Growing Degree Days, $GDD = \sum \max(0, \frac{T_{\max} + T_{\min}}{2} - T_{\text{base}})$) and atmospheric evaporative demand (Vapor Pressure Deficit, $VPD$).
   - We formulate a non-linear spatiotemporal progression model (`PhenologyDiseaseTrajectoryNet`) tracking disease severity across growth stages (V4 vegetative $\to$ R1 flowering $\to$ R3 grain fill $\to$ R6 maturity).
   - We simulate and mathematically quantify the **Area Under the Disease Progress Curve (AUDPC)** and demonstrate that early Variable-Rate Application (VRA) intervention at $t \le 7\text{ days}$ (Stage 1 Pre-Visual) saves $>78\%$ of potential canopy damage and generates **$+\$1,566.50\text{/ha}$** in net revenue compared to unchecked epidemics ($\$924.50\text{/ha}$).

2. **Real-Time High-Throughput RESTful FastAPI Microservice (`app/api.py`):**
   - Engineered an asynchronous, high-throughput REST API serving real-time edge payloads from tractor sprayers and drone fleets:
     - `GET /health` — Hardware target and cache heartbeat.
     - `GET /model/metadata` — Model taxonomy and IPM agronomic thresholds.
     - `POST /predict/pixel` — Sub-millisecond single-pixel / spectrometer probe endpoint with calibrated reflectance, 8 biophysical indices, DSI, conformal $95\%$ bounds, and VRA spray dosage.
     - `POST /predict/batch` — High-throughput streaming batch spatial tile inference.
     - `POST /forecast/trajectory` — Spatiotemporal multi-day disease progression simulator under microclimate scenarios.
     - `POST /scouting/optimize` — Dynamic traveling field scout route solver returning ordered GPS-referenced parcel waypoints.
     - `POST /governance/drift-audit` — Real-time sensor drift & distribution stability audit.

3. **MLOps Spectral Sensor Drift & Population Stability Index (PSI) Governance:**
   - Continuous auditing of spectrometer optical degradation, front-lens dust accumulation, and atmospheric aerosol attenuation.
   - Computes decile-binned **Population Stability Index (PSI)** and **Wasserstein-1 Distance** across 200 spectral channels, enforcing automated alert policies:
     - $\text{PSI} < 0.10$: Nominal Calibration (Continue edge real-time serving).
     - $0.10 \le \text{PSI} < 0.25$: Moderate Drift (Trigger automated sensor dark-current zeroing).
     - $\text{PSI} \ge 0.25$: Severe Drift (Halt autonomous spray; dispatch field agronomist).

4. **Grand Capstone 10-Day Synthesis:**
   - Synthesis of 10 consecutive engineering milestones demonstrating how TerraSpectra overcame the remote sensing spatial generalization cliff, achieved pre-visual early warning ($\text{ROC-AUC} = 0.9750$), cut chemical usage by **$62.4\%$**, and saved **$\$63,070\text{ USD}$** across $409\text{ ha}$ of agricultural farmland.

In [ ]:
import os
import sys
import json
import time
import math
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# Ensure repository root is on sys.path
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
PROC_DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
FIG_DIR = os.path.join(BASE_DIR, "reports", "figures")

print("Environment & Directory Paths Initialized Successfully.")

### 2. Calibrated Spectrometer Ingestion & Baseline Data

In [ ]:
mat_path = os.path.join(RAW_DATA_DIR, "Indian_pines_corrected.mat")
gt_path = os.path.join(RAW_DATA_DIR, "Indian_pines_gt.mat")

cube_raw = sio.loadmat(mat_path)["indian_pines_corrected"].astype(np.float32)
gt = sio.loadmat(gt_path)["indian_pines_gt"].astype(np.int32)
cube_refl = np.clip(cube_raw / 10000.0, 0.0, 1.0)
H, W, B = cube_refl.shape

df_waves = pd.read_csv(os.path.join(PROC_DATA_DIR, "indian_pines_wavelengths.csv"))
clean_wl = df_waves[df_waves["status"] == "Retained (Clean Signal)"]["wavelength_nm"].values
wavelengths = clean_wl if len(clean_wl) == B else np.linspace(400.0, 2400.0, B)

print(f"Loaded Hyperspectral Tensor: {H} x {W} x {B} channels ({wavelengths[0]:.1f} - {wavelengths[-1]:.1f} nm)")

### 3. Multi-Temporal Phenology & Disease Progression Dynamics
Pathogen proliferation is formulated as a coupled non-linear differential equation:
$$\frac{\partial DSI(t)}{\partial t} = r(GDD, VPD) \cdot DSI(t) \cdot (1 - DSI(t)) - \gamma(t) \cdot VRA(t)$$
where $r(GDD, VPD) = r_{\text{base}} \cdot \frac{VPD}{1.5}$ and $\gamma(t)$ is the curative chemical decay rate.

In [ ]:
df_forecast = pd.read_csv(os.path.join(PROC_DATA_DIR, "day10_multitemporal_forecast_trajectories.csv"))
df_forecast.head(14)

In [ ]:
# Display Generated Figure 1: Multi-Temporal Phenology & Disease Progression
from IPython.display import Image
Image(filename=os.path.join(FIG_DIR, "day10_multitemporal_phenology_and_disease_forecasting.png"))

### 4. Production High-Throughput RESTful FastAPI Microservice
We test all core endpoints of the real-time inference microservice (`app/api.py`) using `fastapi.testclient.TestClient`:
- `GET /health` & `GET /model/metadata`
- `POST /predict/pixel` (sub-millisecond tractor boom sensor inference)
- `POST /predict/batch` (vectorized tile streaming)
- `POST /forecast/trajectory` (dynamic multi-day simulation)
- `POST /scouting/optimize` (optimal agronomist inspection route)

In [ ]:
from fastapi.testclient import TestClient
from app.api import app

client = TestClient(app)

# Health & Metadata
res_health = client.get("/health").json()
print("Microservice Health Status:", json.dumps(res_health, indent=2))

# Sub-millisecond Pixel Inference Probe
test_spec = cube_refl[72, 72, :].tolist()
res_pix = client.post("/predict/pixel", json={"reflectance": test_spec, "crop_type": "Soybean-mintill"}).json()
print("\nReal-Time Pixel Inference Result:")
print(f"Predicted Stage: {res_pix['predicted_stage_name']}")
print(f"Disease Severity Index (DSI): {res_pix['disease_severity_index']:.4f}")
print(f"Conformal 95% Confidence Interval: {res_pix['conformal_95_ci']}")
print(f"Recommended VRA Dosage: {res_pix['vra_spray_dosage_lha']} L/ha")
print(f"Inference Latency: {res_pix['inference_latency_ms']:.2f} ms")

In [ ]:
df_api_bench = pd.read_csv(os.path.join(PROC_DATA_DIR, "day10_api_performance_benchmarks.csv"))
df_api_bench

In [ ]:
# Display Generated Figure 2: FastAPI Microservice Production Architecture
Image(filename=os.path.join(FIG_DIR, "day10_fastapi_production_microservice_architecture.png"))

### 5. MLOps Sensor Drift & Population Stability Index (PSI) Governance
To maintain high diagnostic reliability in outdoor farm environments, spectrometer data is audited using decile-binned Population Stability Index (PSI) and Wasserstein-1 distances to detect optic dust, aerosol haze, and hardware calibration shifts.

In [ ]:
df_drift = pd.read_csv(os.path.join(PROC_DATA_DIR, "day10_sensor_drift_governance_metrics.csv"))
df_drift

In [ ]:
# Display Generated Figure 3: MLOps Sensor Drift & Model Governance
Image(filename=os.path.join(FIG_DIR, "day10_mlops_sensor_drift_and_model_governance.png"))

### 6. Grand Capstone 10-Day Research Synthesis & Defense
Over 10 days of rigorous research and engineering, TerraSpectra delivered an end-to-end, physics-grounded, production-ready precision agricultural intelligence system.

In [ ]:
with open(os.path.join(PROC_DATA_DIR, "day10_capstone_grand_summary.json"), "r") as f:
    capstone_summary = json.load(f)

print(json.dumps(capstone_summary, indent=2))

In [ ]:
# Display Generated Figure 4: Grand 10-Day Capstone Synthesis & ROI
Image(filename=os.path.join(FIG_DIR, "day10_grand_10day_capstone_synthesis_and_roi.png"))